# Agentic AI-Based Legal Document Processing System

**Project:** Intelligent multi-agent system for automated document classification and processing

**Framework:** Google ADK

**Business Problem:** Automate classification of cease & desist requests to reduce manual processing time and errors

---

## Table of Contents

1. [System Requirements](#system-requirements)
2. [Installation & Setup](#installation--setup)
3. [Database Configuration](#database-configuration)
4. [Multi-Agent Architecture](#multi-agent-architecture)
5. [System Execution](#system-execution)
6. [Results & Analysis](#results--analysis)
7. [Implementation Verification](#implementation-verification)

---

## System Requirements

### Core Functionality
- Classify Documents into 3 categories: "Cease", "Uncertain", "Irrelevant"
- Process Based on Classification with specialized agents
- Human-in-the-Loop (HITL) for uncertain cases
- Database Interaction for cease requests
- Auditing for complete compliance trail
- Multiple Agents working together

### Technical Requirements
- Python 3.8+
- LiteLlm
- Google Gemini AI API access
- SQLite database
- PDF processing libraries (PyMuPDF or pdfplumber)
- SQLAlchemy ORM

## 1. Installation & Setup

In [ ]:
# Core imports
import os
import json
import re
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Optional

# AI and Database
from google.adk import Agent
from google.adk.models.lite_llm import LiteLlm
from dotenv import load_dotenv

from sqlalchemy import create_engine, Column, Integer, String, Text, Float
from sqlalchemy.orm import declarative_base, sessionmaker

# PDF processing with fallback
try:
    import fitz  # PyMuPDF
    PDF_LIB = "fitz"
except ImportError:
    try:
        import pdfplumber
        PDF_LIB = "pdfplumber"
    except ImportError:
        PDF_LIB = None
        print("Warning: No PDF library available")

# Load environment variables from .env file
# load_dotenv(os.path.join(os.path.dirname(__file__), ".env"))

# GROQ_API_KEY = "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"

# Configure Groq AI via LiteLLM
model = LiteLlm(
    model="groq/llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
)

print("System initialized")
print(f"PDF library: {PDF_LIB}")
print(f"Model: groq/llama-3.1-8b-instant")



System initialized
PDF library: fitz
Model: groq/llama-3.1-8b-instant


## 2. Database Configuration

In [2]:
# Database models
Base = declarative_base()

class CeaseRequest(Base):
    """Database model for cease & desist requests"""
    __tablename__ = 'cease_requests'
    id = Column(Integer, primary_key=True)
    filename = Column(String)
    doc_type = Column(String)
    confidence = Column(Float)
    summary = Column(Text)
    extracted_details = Column(Text)  # JSON string
    date_received = Column(String)
    created_at = Column(String)

# Initialize database
engine = create_engine('sqlite:///cease_requests.db', echo=False)
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session = Session()

print("Database ready for cease requests")

Database ready for cease requests


## 3. Multi-Agent Architecture

### Agent Architecture Overview
```
Manager Agent (Coordinator)
├── Classification Agent (AI-powered analysis)
├── Database Agent (Cease request storage)
├── Archiving Agent (Irrelevant document handling)
├── HITL Agent (Human review workflow)
└── Audit Agent (Compliance logging)
```

### Dependencies Installation

Before running the system, ensure you have the required packages:

```bash
pip install google-adk python-dotenv sqlalchemy pdfplumber
# Optional: pip install PyMuPDF for faster PDF processing
```

### Specialized Agents Implementation

### 3.1 Document Loader Agent

In [3]:
import fitz  # PyMuPDF
from PIL import Image
import pytesseract

class DocumentLoaderAgent:
    """Agent responsible for loading and extracting text from documents"""

    def __init__(self):
        self.name = "DocumentLoader"

    def load_document(self, file_path: str) -> Optional[str]:
        """Load and extract text from PDF documents with multiple fallback methods"""
        if not os.path.exists(file_path):
            print(f" {self.name}: File not found: {file_path}")
            return None

        # Method 1: PyMuPDF (fastest)
        if PDF_LIB == "fitz":
            try:
                doc = fitz.open(file_path)
                text = ""
                for page in doc:
                    text += page.get_text() + "\n"
                doc.close()
                print(f" {self.name}: Loaded {os.path.basename(file_path)} with PyMuPDF")
                return text.strip()
            except Exception as e:
                print(f" {self.name}: PyMuPDF failed: {e}")

        # Method 2: pdfplumber (good OCR support)
        if PDF_LIB == "pdfplumber":
            try:
                with pdfplumber.open(file_path) as pdf:
                    text = ""
                    for page in pdf.pages:
                        page_text = page.extract_text()
                        if page_text:
                            text += page_text + "\n"
                    print(f" {self.name}: Loaded {os.path.basename(file_path)} with pdfplumber")
                    return text.strip()
            except Exception as e:
                print(f" {self.name}: pdfplumber failed: {e}")

         # Method 3: OCR (for scanned PDFs)
        try:
            print(f"{self.name}: Running OCR...")

            doc = fitz.open(file_path)
            ocr_text = ""

            for page in doc:
                pix = page.get_pixmap(dpi=300)
                #pix = doc[page.number].get_pixmap(dpi=300)
                img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

                page_text = pytesseract.image_to_string(img, lang="eng")
                ocr_text += page_text + "\n"
                img = Image.open("page0.png")
                print(f" {self.name}: OCR Text: {pytesseract.image_to_string(img)}")

            doc.close()

            if ocr_text.strip():
                print(f"{self.name}: OCR extraction successful")
                print(f"OCR Text length: {len(ocr_text)}")
                return ocr_text.strip()

        except Exception as e:
            print(f"{self.name}: OCR failed: {e}")

        print(f"{self.name}: All methods failed")

        # Method 4: Basic text read
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read().strip()
            if text:
                print(f" {self.name}: Loaded {os.path.basename(file_path)} as plain text")
                return text
        except:
            pass

        print(f" {self.name}: All loading methods failed for {os.path.basename(file_path)}")
        return None

### 3.2 Classification Agent

In [ ]:
import litellm
#litellm.api_key = "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"
class ClassificationAgent:
    """AI-powered agent for document classification"""

    def __init__(self):
        self.name = "ClassificationAgent"

    def classify_document(self, text: str) -> Dict[str, Any]:
        """Classify document using Groq LLM"""
        print(f"[Classification] Received text length: {len(text) if text else 0}")

        if text:
            snippet = text[:120].replace("\n", " ")
            print(f"[Classification] Text snippet: {snippet}")

        if not text or len(text.strip()) < 10:
            print("[Classification] Insufficient content; returning Uncertain")
            return {"label": "Uncertain", "confidence": 0.3, "reason": "Insufficient text content"}

        prompt = f"""
        Analyze this document and classify it into ONE of these categories:

        1. "Cease" - Valid cease & desist request
        2. "Uncertain" - unclear or borderline
        3. "Irrelevant" - unrelated content

        Document text:
        {text[:3000]}

        Return ONLY JSON:
        {{"label": "Cease", "confidence": 0.95, "reason": "brief explanation"}}
        """

        try:
            snippet = prompt[:120].replace("\n", " ")
            print(f"[Classification] Sending prompt to Groq; first 120 chars: {snippet}")

            # FIX: use generate() instead of completion()
            #response = model.generate(prompt)

            response = litellm.completion(
                model="groq/llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=120,
                temperature=0.1
                )

            output_text = response['choices'][0]['message']['content']

            # FIX: extract text properly
            #output_text = response.text

            snippet = output_text[:240].replace("\n", " ")
            print(f"[Classification] Groq raw output snippet: {snippet}")

            # Extract JSON
            json_match = re.search(r'\{.*\}', output_text, re.DOTALL)
            if json_match:
                data = json.loads(json_match.group())

                valid_labels = ["Cease", "Uncertain", "Irrelevant"]
                if data.get("label") not in valid_labels:
                    data["label"] = "Uncertain"
                    data["confidence"] = 0.5
                    data["reason"] = "Invalid classification label"

                print(f"[Classification] Parsed classification: {data}")
                return data

        except Exception as e:
            print(f"[Classification] Classification error: {e}")

        return {"label": "Uncertain", "confidence": 0.5, "reason": "Classification failed"}

### 3.3 Database Agent

In [5]:
class DatabaseAgent:
    """Agent responsible for database operations for cease requests"""

    def __init__(self, db_session):
        self.name = "DatabaseAgent"
        self.session = db_session

    def store_cease_request(self, filename: str, classification: Dict[str, Any], extracted_details: Dict = None) -> bool:
        """Store validated cease request in database"""
        print(f"[Database] store_cease_request: filename={filename}, classification={classification}")
        try:
            record = CeaseRequest(
                filename=filename,
                doc_type="Cease",
                confidence=classification["confidence"],
                summary=classification["reason"],
                extracted_details=json.dumps(extracted_details or {}),
                date_received=datetime.now().isoformat(),
                created_at=datetime.now().isoformat()
            )
            self.session.add(record)
            self.session.commit()
            print(f"[Database] Stored cease request '{filename}' (ID: {record.id})")
            return True
        except Exception as e:
            self.session.rollback()
            print(f"[Database] Failed to store cease request: {e}")
            return False

### 3.4 Archiving Agent

In [6]:
class ArchivingAgent:
    """Agent responsible for archiving irrelevant documents"""

    def __init__(self):
        self.name = "ArchivingAgent"

    def archive_irrelevant(self, filename: str, classification: Dict[str, Any]) -> bool:
        """Archive irrelevant document to flat file"""
        print(f"[Archive] archive_irrelevant: filename={filename}, classification={classification}")
        try:
            archive_path = "data/archive_log.json"
            os.makedirs("data", exist_ok=True)

            # Load existing archive
            archive = []
            if os.path.exists(archive_path):
                with open(archive_path, 'r') as f:
                    archive = json.load(f)

            # Add new entry
            archive.append({
                "filename": filename,
                "reason": classification["reason"],
                "confidence": classification["confidence"],
                "date_archived": datetime.now().isoformat()
            })

            # Save updated archive
            with open(archive_path, 'w') as f:
                json.dump(archive, f, indent=2)

            print(f"[Archive] Archived irrelevant document '{filename}'. total archived={len(archive)}")
            return True
        except Exception as e:
            print(f"[Archive] Failed to archive document: {e}")
            return False

### 3.5 HITL Agent

In [7]:
class HITLAgent:
    """Human-in-the-Loop agent for uncertain cases requiring manual review"""

    def __init__(self):
        self.name = "HITLAgent"

    def queue_for_review(self, filename: str, classification: Dict[str, Any], text_preview: str) -> bool:
        """Queue uncertain document for human review"""
        print(f"[HITL] queue_for_review: filename={filename}, classification={classification}")
        try:
            queue_path = "data/hitl_queue.json"
            os.makedirs("data", exist_ok=True)

            # Load existing queue
            queue = {"pending": [], "reviewed": []}
            if os.path.exists(queue_path):
                with open(queue_path, 'r') as f:
                    queue = json.load(f)

            # Add to pending reviews
            review_id = f"HITL_{datetime.now().timestamp()}"
            queue["pending"].append({
                "review_id": review_id,
                "filename": filename,
                "confidence": classification["confidence"],
                "reason": classification["reason"],
                "text_preview": text_preview[:500],
                "date_queued": datetime.now().isoformat()
            })

            # Save updated queue
            with open(queue_path, 'w') as f:
                json.dump(queue, f, indent=2)

            print(f"[HITL] Queued '{filename}' for human review (ID: {review_id}); pending={len(queue['pending'])}")
            return True
        except Exception as e:
            print(f"[HITL] Failed to queue for review: {e}")
            return False

### 3.6 Audit Agent

In [8]:
class AuditAgent:
    """Agent responsible for auditing all system activities"""

    def __init__(self):
        self.name = "AuditAgent"

    def log_activity(self, filename: str, action: str, classification: str, destination: str, details: Dict = None) -> bool:
        """Log all system activities for compliance"""
        print(f"[Audit] log_activity: filename={filename}, action={action}, classification={classification}, destination={destination}, details={details}")
        try:
            audit_path = "data/audit_log.json"
            os.makedirs("data", exist_ok=True)

            # Load existing audit log
            audit_log = []
            if os.path.exists(audit_path):
                with open(audit_path, 'r') as f:
                    audit_log = json.load(f)

            # Add new audit entry
            audit_log.append({
                "timestamp": datetime.now().isoformat(),
                "filename": filename,
                "action": action,
                "classification": classification,
                "destination": destination,
                "details": details or {},
                "agent": self.name
            })

            # Save updated audit log
            with open(audit_path, 'w') as f:
                json.dump(audit_log, f, indent=2)

            print(f"[Audit] Logged {action} for '{filename}' → {destination}; total entries={len(audit_log)}")
            return True
        except Exception as e:
            print(f"[Audit] Failed to log activity: {e}")
            return False

### 3.7 Manager Agent (Coordinator)

In [9]:
import time

class ManagerAgent:
    """Main orchestrator agent that coordinates all specialized agents"""

    def __init__(self):
        self.name = "ManagerAgent"

        # Initialize specialized agents
        self.loader = DocumentLoaderAgent()
        self.classifier = ClassificationAgent()
        self.db_agent = DatabaseAgent(session)
        self.archive_agent = ArchivingAgent()
        self.hitl_agent = HITLAgent()
        self.audit_agent = AuditAgent()

        print(" ManagerAgent: All specialized agents initialized and ready")

    def process_document(self, file_path: str) -> Dict[str, Any]:
        """Complete document processing workflow"""
        filename = os.path.basename(file_path)

        print(f"\n {self.name}: Processing document '{filename}'")
        print("=" * 60)

        # Phase 1: Load Document
        print(f" Phase 1: Loading document...")
        text = self.loader.load_document(file_path)

        if not text:
            result = {
                "filename": filename,
                "success": False,
                "error": "Failed to load document",
                "destination": "failed"
            }
            self.audit_agent.log_activity(filename, "load_failed", "Unknown", "failed")
            return result

        # Phase 2: Classify Document
        print(f" Phase 2: Classifying document...")
        classification = self.classifier.classify_document(text)

        # Phase 3: Route Based on Classification
        print(f" Phase 3: Routing to {classification['label'].lower()} workflow...")

        success = False
        destination = "unknown"

        if classification["label"] == "Cease":
            # Store in database
            success = self.db_agent.store_cease_request(filename, classification)
            destination = "database"

        elif classification["label"] == "Irrelevant":
            # Archive to flat file
            success = self.archive_agent.archive_irrelevant(filename, classification)
            destination = "archive"

        else:  # Uncertain
            # Queue for human review
            text_preview = text[:500] if text else "No text available"
            success = self.hitl_agent.queue_for_review(filename, classification, text_preview)
            destination = "hitl"

        # Phase 4: Audit Everything
        print(f" Phase 4: Auditing activity...")
        self.audit_agent.log_activity(
            filename,
            "process_document",
            classification["label"],
            destination,
            {"confidence": classification["confidence"], "reason": classification["reason"]}
        )

        result = {
            "filename": filename,
            "classification": classification["label"],
            "confidence": classification["confidence"],
            "destination": destination,
            "success": success
        }

        status = "" if success else ""
        print(f"{status} Processing complete: {filename} → {destination}")
        print("=" * 60)

        return result

    def process_folder(self, folder_path: str) -> List[Dict[str, Any]]:
        """Process all PDFs in a folder"""
        if not os.path.exists(folder_path):
            print(f" Folder not found: {folder_path}")
            return []

        pdf_files = list(Path(folder_path).glob("*.pdf"))
        print(f" Found {len(pdf_files)} PDF documents to process")

        results = []
        for i, pdf_file in enumerate(pdf_files, 1):
            print(f"\n Processing {i}/{len(pdf_files)}...")
            result = self.process_document(str(pdf_file))
            results.append(result)

            time.sleep(2)

        return results

    def get_system_stats(self) -> Dict[str, Any]:
        """Get comprehensive system statistics"""
        stats = {
            "cease_requests": session.query(CeaseRequest).count(),
            "archive_count": 0,
            "hitl_pending": 0,
            "audit_entries": 0
        }

        # Count archived documents
        try:
            if os.path.exists("data/archive_log.json"):
                with open("data/archive_log.json", 'r') as f:
                    stats["archive_count"] = len(json.load(f))
        except:
            pass

        # Count pending HITL reviews
        try:
            if os.path.exists("data/hitl_queue.json"):
                with open("data/hitl_queue.json", 'r') as f:
                    queue = json.load(f)
                    stats["hitl_pending"] = len(queue.get("pending", []))
        except:
            pass

        # Count audit entries
        try:
            if os.path.exists("data/audit_log.json"):
                with open("data/audit_log.json", 'r') as f:
                    stats["audit_entries"] = len(json.load(f))
        except:
            pass

        return stats

print(" Multi-Agent System ready!")

 Multi-Agent System ready!


## 4. System Execution

### Usage Instructions

1. Ensure all dependencies are installed
2. Place PDF documents in the `data/` folder
3. Run the notebook cells in order
4. Review the processing results and system statistics

### Execution Output

In [10]:
# Initialize the complete multi-agent system
manager = ManagerAgent()
results = manager.process_folder("data")
print("DEBUG results:", results)

print(" STARTING CEASE & DESIST DOCUMENT PROCESSING SYSTEM")
print("=" * 70)

# Process all documents in the data folder
results = manager.process_folder("data")
print(results)

# Generate comprehensive processing summary
print("\n COMPREHENSIVE PROCESSING SUMMARY")
print("=" * 70)

total = len(results)
successful = sum(1 for r in results if r.get("success", False))
failed = total - successful

# Classification breakdown
cease_count = sum(1 for r in results if r.get("classification") == "Cease")
uncertain_count = sum(1 for r in results if r.get("classification") == "Uncertain")
irrelevant_count = sum(1 for r in results if r.get("classification") == "Irrelevant")

# Destination breakdown
database_count = sum(1 for r in results if r.get("destination") == "database")
archive_count = sum(1 for r in results if r.get("destination") == "archive")
hitl_count = sum(1 for r in results if r.get("destination") == "hitl")

print(f" Total Documents Processed: {total}")
print(f" Successful Processing: {successful}")
print(f" Failed Processing: {failed}")
print()

print(" CLASSIFICATION RESULTS:")
print(f"  • Cease Requests: {cease_count} documents")
print(f"  • Uncertain Cases: {uncertain_count} documents")
print(f"  • Irrelevant Docs: {irrelevant_count} documents")
print()

print(" PROCESSING DESTINATIONS:")
print(f"  • Database (Cease): {database_count} documents")
print(f"  • Archive (Irrelevant): {archive_count} documents")
print(f"  • HITL Review (Uncertain): {hitl_count} documents")
print()

# Show detailed results
if results:
    print(" DETAILED RESULTS:")
    print("-" * 50)
    for i, result in enumerate(results[:5], 1):  # Show first 5
        status = "" if result.get("success") else ""
        classification = result.get("classification", "Unknown")
        confidence = result.get("confidence", 0)
        destination = result.get("destination", "unknown")

        print(f"{i}. {status} {result['filename']}")
        print(f"   Classification: {classification} ({confidence:.2f})")
        print(f"   Destination: {destination}")
        print()

# System-wide statistics
print(" SYSTEM STATISTICS:")
print("-" * 50)
stats = manager.get_system_stats()
print(f" Database Records: {stats['cease_requests']} cease requests")
print(f" Archived Documents: {stats['archive_count']} irrelevant docs")
print(f" Pending Reviews: {stats['hitl_pending']} documents")
print(f" Audit Trail: {stats['audit_entries']} logged activities")

print("\n CEASE & DESIST PROCESSING SYSTEM EXECUTION COMPLETE!")
print("=" * 70)

# Demonstrate all required capabilities
print("\n IMPLEMENTATION VERIFICATION:")
print("   Multiple Agents: Classification, Database, Archive, HITL, Audit")
print("   Human-in-the-Loop: Uncertain cases queued for review")
print("   Database Interaction: Cease requests stored with full details")
print("   Auditing: Complete compliance trail maintained")
print("   3-Category Classification: Cease, Uncertain, Irrelevant")
print("=" * 70)

 ManagerAgent: All specialized agents initialized and ready
 Found 29 PDF documents to process

 Processing 1/29...

 ManagerAgent: Processing document '01_copyright_infringement_photography.pdf'
 Phase 1: Loading document...
 DocumentLoader: Loaded 01_copyright_infringement_photography.pdf with PyMuPDF
 Phase 2: Classifying document...
[Classification] Received text length: 3153
[Classification] Text snippet: CEASE AND DESIST LETTER Priya Nair Professional Photographer Priya Nair Photography LLC 482 Oak Ridge Drive Austin, TX 7
[Classification] Sending prompt to Groq; first 120 chars:          Analyze this document and classify it into ONE of these categories:          1. "Cease" - Valid cease & desist 
[Classification] Groq raw output snippet: {   "label": "Cease",   "confidence": 0.95,   "reason": "The document is a formal cease and desist letter sent by Priya Nair to David Chen, demanding that he and/or his organization immediately stop using Priya Nair's copyrighted photograph
[Cl

## 5. Results & Analysis

### Processing Summary
After execution, the system provides comprehensive statistics including:
- Total documents processed
- Classification breakdown (Cease/Uncertain/Irrelevant)
- Processing destinations (Database/Archive/HITL)
- Success/failure rates
- System-wide statistics

### Output Files Generated
- `cease_requests.db`: SQLite database with processed cease requests
- `data/archive_log.json`: Archived irrelevant documents
- `data/hitl_queue.json`: Documents awaiting human review
- `data/audit_log.json`: Complete audit trail of all activities

### Key Metrics
- **Accuracy**: Classification confidence scores
- **Throughput**: Documents processed per execution
- **HITL Rate**: Percentage requiring human intervention
- **Audit Coverage**: Complete activity logging

## 6. Implementation Verification

### ✅ Verified Capabilities
- **Multi-Agent System**: Coordinator orchestrates specialized agents
- **AI-Powered Classification**: Gemini AI for document analysis
- **Database Operations**: SQLAlchemy ORM for cease request storage
- **Flat-File Archiving**: JSON-based storage for irrelevant documents
- **Human-in-the-Loop Workflow**: Queue system for uncertain cases
- **Comprehensive Auditing**: Complete compliance trail maintenance
- **Robust Error Handling**: Fallback PDF processing methods
- **Scalable Architecture**: Modular design for future enhancements

### System Architecture Benefits
1. **Modularity**: Each agent has a single responsibility
2. **Extensibility**: Easy to add new agent types or processing logic
3. **Reliability**: Fallback mechanisms ensure processing continuity
4. **Compliance**: Full audit trail for regulatory requirements
5. **Maintainability**: Clean separation of concerns
6. **Monitoring**: Comprehensive logging and statistics

---

**End of Implementation**

In [11]:
agent = ClassificationAgent()

result = agent.classify_document(
    "This is a legal cease and desist notice for copyright violation"
)

print(result)

[Classification] Received text length: 63
[Classification] Text snippet: This is a legal cease and desist notice for copyright violation
[Classification] Sending prompt to Groq; first 120 chars:          Analyze this document and classify it into ONE of these categories:          1. "Cease" - Valid cease & desist 
[Classification] Groq raw output snippet: {"label": "Cease", "confidence": 0.95, "reason": "The document explicitly states it is a legal cease and desist notice for copyright violation, indicating a clear request to stop the infringing activity."}
[Classification] Parsed classification: {'label': 'Cease', 'confidence': 0.95, 'reason': 'The document explicitly states it is a legal cease and desist notice for copyright violation, indicating a clear request to stop the infringing activity.'}
{'label': 'Cease', 'confidence': 0.95, 'reason': 'The document explicitly states it is a legal cease and desist notice for copyright violation, indicating a clear request to stop the infring